# DNN Programming Assignment 2 — Problem 2: Transformer from Scratch

**Student Name:** `Prahlad Vinayak Avadhani`  
**Student ID:** `2026AJ05253`  

---

## Documented Assumptions & Variant Configuration
- **Student ID:** `2026AJ05253` (last digit = `3`, odd).
- **Three Seeds:** `SEEDS = [3, 13, 23]`.
- **Transformer Architecture Defaults:** `d = 128`, `d_ff = 4d = 512`, `num_heads = 4`, `num_layers = 4`, `context_len = 128`.
- **Task 1 (3-Token Hand-Checkable Example):** Uses a 3-token, 2-dimensional example ($n = 3, d_k = 2$) computed step-by-step by hand and verified against code.
- **Forbidden Layers Rule:** `nn.MultiheadAttention` and `nn.Transformer*` are never used in any Problem 2 model; `F.scaled_dot_product_attention` is used strictly inside unit test `assert` checks.

In [ ]:
import os
import time
import math
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset

os.makedirs('figures', exist_ok=True)
RESULTS_CSV_PATH = '../results.csv'
GENERATED_SAMPLES_PATH = 'generated_samples.txt'

STUDENT_ID = '2026AJ05253'
SEEDS = [3, 13, 23]
D_MODEL = 128
D_FF = 512
NUM_HEADS = 4
NUM_LAYERS = 4
CONTEXT_LEN = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

def set_seed(seed: int):
    """Fix all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def log_result(problem, model, seed, seq_len='', test_acc='', macro_f1='',
               test_mse='', params='', sec_per_epoch='', val_loss_nats='',
               bits_per_char='', notes=''):
    """Append a single experiment run row to ../results.csv."""
    columns = [
        'problem', 'model', 'seed', 'seq_len', 'test_acc', 'macro_f1',
        'test_mse', 'params', 'sec_per_epoch', 'val_loss_nats', 'bits_per_char', 'notes'
    ]
    row = pd.DataFrame([{
        'problem': problem,
        'model': model,
        'seed': seed,
        'seq_len': seq_len,
        'test_acc': test_acc,
        'macro_f1': macro_f1,
        'test_mse': test_mse,
        'params': params,
        'sec_per_epoch': sec_per_epoch,
        'val_loss_nats': val_loss_nats,
        'bits_per_char': bits_per_char,
        'notes': notes
    }], columns=columns)
    write_header = not os.path.exists(RESULTS_CSV_PATH) or os.path.getsize(RESULTS_CSV_PATH) == 0
    row.to_csv(RESULTS_CSV_PATH, mode='a', header=write_header, index=False)

---
## Task 1 (P2.1): Scaled Dot-Product Attention from Scratch (10 Points)

- Implement `scaled_dot_product_attention(Q, K, V, mask=None)` using basic tensor operations.
- Unit tests (`assert` checks) immediately following:
  1. Each row of attention weights sums to `1`.
  2. Output matches `torch.nn.functional.scaled_dot_product_attention` within `1e-5`.
  3. Causal mask gives zero weight (`0.0`) to future positions.
  4. Hand-checkable 3-token example ($n = 3, d_k = 2$) computed by hand and matched by code.

In [ ]:
def scaled_dot_product_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, mask: torch.Tensor = None):
    """
    Scaled Dot-Product Attention from scratch.
    Input shapes:
      Q:    (..., n_q, d_k)
      K:    (..., n_k, d_k)
      V:    (..., n_k, d_v)
      mask: broadcastable to (..., n_q, n_k), boolean (True = keep, False = mask out) or additive float
    Output shapes:
      output:       (..., n_q, d_v)
      attn_weights: (..., n_q, n_k)
    """
    # TODO (Phase 2 Step 4): Implement scaled dot-product attention
    pass


In [ ]:
# Unit Tests & 3-Token Hand-Checkable Verification for Task 1 (P2.1)
# TODO (Student): Write assert tests for row-sum=1, agreement with F.scaled_dot_product_attention within 1e-5,
# zero future weights under causal mask, and the 3-token numerical example.


---
## Task 2 (P2.2): Multi-Head Attention, Positional Encoding, and Encoder Block (10 Points)

- Implement `MultiHeadAttention` with `W_Q, W_K, W_V, W_O` and head reshaping.
- Implement `SinusoidalPositionalEncoding`.
- Implement pre-LayerNorm `TransformerEncoderBlock` with feed-forward width $4d$, residual connections, and dropout.
- Derive weight count $4d^2 + 2 \cdot d \cdot d_{\text{ff}} = 196,608$ (for $d=128, d_{\text{ff}}=512$) and reconcile biases (`1,152`) and LayerNorm parameters (`512`) with `sum(p.numel()) = 198,272`.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention from scratch.
    Input shape:  x of shape (B, n, d)
    Output shape: (out, attn_weights) where out is (B, n, d) and attn_weights is (B, num_heads, n, n)
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        # TODO (Phase 2 Step 5): Define W_Q, W_K, W_V, W_O projections and dropout
        pass

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        # TODO (Phase 2 Step 5): Project Q, K, V, split heads, apply scaled_dot_product_attention, concat, project W_O
        pass


class SinusoidalPositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding from scratch.
    Input shape:  x of shape (B, n, d)
    Output shape: (B, n, d)
    """
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        # TODO (Phase 2 Step 6): Precompute sinusoidal positional encoding matrix pe of shape (1, max_len, d_model)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO (Phase 2 Step 6): Add positional encoding to x[:, :x.size(1), :]
        pass


class TransformerEncoderBlock(nn.Module):
    """
    Pre-LayerNorm Transformer Encoder Block from scratch.
    Input shape:  x of shape (B, n, d)
    Output shape: (out, attn_weights) where out is (B, n, d)
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        if d_ff is None:
            d_ff = 4 * d_model
        # TODO (Phase 2 Step 7): Define norm1, mha, norm2, ffn (Linear d->d_ff, GELU/ReLU, Dropout, Linear d_ff->d), dropout
        pass

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None):
        # TODO (Phase 2 Step 7): Pre-LN residual connection for MHA + Pre-LN residual connection for FFN
        pass


In [ ]:
# Unit Tests & Parameter Reconciliation for Task 2 (P2.2)
# TODO (Student): Verify output tensor shapes and reconcile weight count (196,608) + biases (1,152) + LayerNorm (512) = 198,272.


---
## Task 3 (P2.3): Task A — Encoder Classifier on UCI HAR (10 Points)

- Reuse UCI HAR data pipeline from Problem 1.
- Embed each time step (or patch of 8 steps), prepend a learnable `CLS` token, and classify with the custom Transformer encoder.
- Ablate positional encoding: `none`, `sinusoidal`, and `learned` across 3 seeds.
- Compare accuracy, macro-F1, parameter count, and training time with the best Problem 1 recurrent model.

In [ ]:
# TODO (Student): Implement HARTransformerClassifier with CLS token and run positional encoding ablation (none, sinusoidal, learned).


---
## Task 4 (P2.4): Task B — Character-Level Causal Language Model (10 Points)

- Dataset: Tiny Shakespeare (~1.1 MB, 65 unique characters, first 90% train, last 10% validation, no shuffling).
- Architecture: Causal Transformer with `context_len = 128`, `d = 128`, `4` heads, `4` layers.
- Report validation loss in nats and bits per character ($\text{BPC} = \text{val\_loss} / \ln 2$).
- Generate 500 characters at temperatures `0.2`, `0.8`, and `1.5` with top-$k$ sampling and save to `generated_samples.txt`.

In [ ]:
# Tiny Shakespeare Download & Vocabulary Setup (Provided in Assignment Specification)
SHAKESPEARE_PATH = 'tinyshakespeare.txt'
if not os.path.exists(SHAKESPEARE_PATH):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    text = urllib.request.urlopen(url).read().decode('utf-8')
    with open(SHAKESPEARE_PATH, 'w', encoding='utf-8') as f:
        f.write(text)
else:
    with open(SHAKESPEARE_PATH, 'r', encoding='utf-8') as f:
        text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
print(f'Total characters: {len(text):,}, Vocabulary size: {vocab_size}')

# TODO (Student): Build 90%/10% unshuffled train/val split, CausalTransformerLM, training loop,
# learning curve plot (figures/p2_lm_learning_curve.png), and top-k generation at T in {0.2, 0.8, 1.5}.


---
## Task 5 (P2.5): Analysis and Ablations (10 Points)

- **5(a):** Plot attention heatmaps of two heads on one sentence (`figures/p2_attention_heatmaps.png`) and interpret them.
- **5(b):** Head count ablation (`heads = 1, 4, 8`) at fixed $d = 128$ (`figures/p2_heads_ablation.png`).
- **5(c):** Remove the causal mask and show what happens to training loss and text generation (`figures/p2_no_mask_ablation.png`).
- **5(d):** Measure time and memory at context lengths `64, 128, 256, 512` (`figures/p2_context_scaling.png`) and relate growth to $O(n^2 d)$.

In [ ]:
# TODO (Student): 5(a) Attention heatmaps of 2 heads on a sample sentence.


In [ ]:
# TODO (Student): 5(b) Heads = 1, 4, 8 ablation at fixed d = 128.


In [ ]:
# TODO (Student): 5(c) Remove causal mask study (training loss leakage vs broken generation).


In [ ]:
# TODO (Student): 5(d) Context length scaling {64, 128, 256, 512} time and memory vs O(n^2 d).
